# Analyse Helios Simulations
This notebook walks through the process of analysing helios simulations which follows 3 main steps:

    (1) Data Preparation
        This stage will setup the project directory, setup expected schemas for dataframes (both dask and pandas), and ultimately read in the helios data and prepare the required per ray information into a .parquet output.
        It will also setup the reference dataset for voxels for each voxel_size in the project (i.e. unique voxel_ids etc.).
    
    (2) Voxel-Ray Intersection
        With valid rays saved per leg of the scan, in the previous step, the goal now is to check ray intersections in all voxels. This will record important information, such as the entry/exit/hit coordinates of the ray which will later be used to gather metrics.
        The main reason these metrics are not gathered yet, is that this stage will remain separate per leg. That way, the metrics can be computed from different combinations of helios legs without re-computing voxel-ray intersections.

    (3) Compute Metrics
        Taking a given set of legs and voxel_sizes, the voxel_ray intersection files will be used to calculate metrics for each voxel, in this case resulting in all outputs from each investigated method.

# Step 1 - Setup Project
Set project paths here

In [2]:
import os

# Set up the project directory
project_dir = '/home/capheus/projects/tumba_fix'
helios_dir = os.path.join(project_dir, 'helios')
references_dir = os.path.join(project_dir, 'references')
results_dir = os.path.join(project_dir, 'results')
valid_rays_dir = os.path.join(project_dir, 'valid_rays')

if not os.path.exists(helios_dir) or not os.path.exists(references_dir):
    raise FileNotFoundError("The specified directories do not exist. Please check the paths.")

if not os.path.exists(valid_rays_dir):
    os.makedirs(valid_rays_dir, exist_ok=True)

if not os.path.exists(results_dir):
    os.makedirs(results_dir, exist_ok=True)

## Step 1 - Data Preparation
This step focuses on converting helios simulation outputs, saving only valid rays into a more efficient .parquet file format.

It expects the following input and will add a new folder (valid_rays) to store all resulting .parquet files.

INPUT:
    project_dir/
    ├── reference/
    │   ├── "{project}_voxel_size_0.2.csv"
    │   ├── "{project}_voxel_size_0.5.csv"
    │   ...
    │   └── "{project}_voxel_size_{v}.csv"
    ├── helios/
    │   ├── "leg000_points.xyz"
    │   ├── "leg000_pulse.txt"
    │   ├── "leg000_fullwave.txt"
    │   ├── "leg001_points.xyz"
    │   ├── "leg001_pulse.txt"
    │   ├── "leg001_fullwave.txt"
    │   ├── ...
    │   ├── "leg{l}_points.xyz"
    │   ├── "leg{l}_pulse.txt"
    │   └── "leg{l}_fullwave.txt"

OUTPUT:
    └── valid_rays/
        ├── "leg_000_valid_rays.parquet"
        ├── "leg_001_valid_rays.parquet"
        ...
        └── "leg_{l}_valid_rays.parquet"

In [1]:
from utils import prepare_helios_data

# Establish the object IDs for the leaves
leaf_object_ids = [0]
wood_object_ids = [1]

# Run the data preparation script
prepare_helios_data(
    input_dir=helios_dir, 
    output_dir=valid_rays_dir, 
    references_dir=references_dir, 
    leaf_object_ids=leaf_object_ids, 
    wood_object_ids=wood_object_ids, 
    use_class=False,
    debug=True
)

NameError: name 'helios_dir' is not defined

### Step 1.5 -  Compute Normals and Weights for Leaf Points

In [3]:
from utils import add_normals_weights_to_valid_rays

# Calculate normals and weights by loading valid rays
add_normals_weights_to_valid_rays(
    valid_rays_dir, 
    debug=True,
    knn=6
)

Adding normals and weights to 2 files...
[########################################] | 100% Completed | 2.94 ss
Initialising voxels


Processing voxels: 100%|██████████| 87/87 [00:02<00:00, 35.91it/s]


Saving results...
Debugging enabled:
         leg_id   ray_id   origin_x   origin_y  origin_z  direction_x  \
3214066       0  3270418  33.882401  75.590103       3.5      -0.0848   
3214094       0  3269502  33.882401  75.590103       3.5      -0.0839   
3214097       0  3270419  33.882401  75.590103       3.5      -0.0847   
3214119       0  3269503  33.882401  75.590103       3.5      -0.0839   
3214145       0  3269504  33.882401  75.590103       3.5      -0.0838   

         direction_y  direction_z    point_x    point_y  point_z  \
3214066      -0.8817       0.4640  31.930300  55.294998  14.1808   
3214094      -0.8649       0.4949  31.933599  55.511700  14.9896   
3214097      -0.8810       0.4655  31.931601  55.308399  14.2172   
3214119      -0.8641       0.4964  31.941000  55.588001  14.9905   
3214145      -0.8632       0.4978  31.964600  55.830898  14.8952   

         echo_intensity  return_number  number_of_returns  normal_x  normal_y  \
3214066    2.022384e+06           

/home/capheus/PlantDensityAnalysis/utils.py:3098: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  print(f"Saved {len(output_files)} valid rays files with normals and weights.")


## Step 2 - Voxel Ray Intersections
This code uses the valid rays from before, alongside the reference datasets in order to create a supporting parquet in the valid rays folder using the voxel_size_{voxel_size}_leg_{leg}_intersections.parquet format.

In [ ]:
from utils import voxel_ray_intersections

# Run intersections
voxel_ray_intersections(
    valid_rays_dir=valid_rays_dir, 
    references_dir=references_dir
)

Initialising leg 0 - 40 partitions, 6746336 rays, 3042 voxels, 172552 rays/partition, 77 voxels/chunk, 39 chunks
Initialising leg 1 - 40 partitions, 6746359 rays, 3042 voxels, 172519 rays/partition, 77 voxels/chunk, 39 chunks
Processing leg 0...
[#########################               ] | 62% Completed | 345.26 sProcess ac49a4692406462aa166a69186681faf. Returning results...
[##########################              ] | 65% Completed | 363.37 sProcess 63ed0286ced54cb0a234a7f5a91bb4a6. Returning results...
[##########################              ] | 67% Completed | 365.32 sProcess 09dfedda83ee4804b7d6ead8923063ab. Returning results...
[###########################             ] | 69% Completed | 370.51 sProcess de6498a560a64e799cd80de49ee47025. Returning results...
[##############################          ] | 75% Completed | 405.28 sProcess 3689af65416e4ecb80d21a3c3609b52a. Returning results...
[########################################] | 100% Completed | 10m 35s
Saving results for leg 0

/home/capheus/PlantDensityAnalysis/utils.py:2159: RuntimeWarning: Mean of empty slice
  valid_path_length_mask = unbound | current_hit | yet_to_hit
/home/capheus/PlantDensityAnalysis/utils.py:2158: RuntimeWarning: Mean of empty slice
  # Exclude rays that have no current hits or yet_to_hits
/home/capheus/PlantDensityAnalysis/utils.py:2159: RuntimeWarning: Mean of empty slice
  valid_path_length_mask = unbound | current_hit | yet_to_hit
/home/capheus/PlantDensityAnalysis/utils.py:2159: RuntimeWarning: Mean of empty slice
  valid_path_length_mask = unbound | current_hit | yet_to_hit
/home/capheus/PlantDensityAnalysis/utils.py:2159: RuntimeWarning: Mean of empty slice
  valid_path_length_mask = unbound | current_hit | yet_to_hit
/home/capheus/PlantDensityAnalysis/utils.py:2158: RuntimeWarning: Mean of empty slice
  # Exclude rays that have no current hits or yet_to_hits
/home/capheus/PlantDensityAnalysis/utils.py:2159: RuntimeWarning: Mean of empty slice
  valid_path_length_mask = unbound

## Step 3 - Compute Metrics
Using the leg_{leg_id}_voxel_size_{voxel_size}_intersections.parquet files (which feature a standardised structure of columns from various inputs), compute the desired metrics and save outputs.

### Step 3.1 - Multi-return
Use the get_voxel_metrics_multireturn function to handle mutli return projects.

In [ ]:
import os
import glob
import utils
import pandas as pd
from utils import calculate_lambda_1, get_voxel_metrics

# Select the desired legs and voxel_sizes to include in the analysis
# Use the shortcut string 'all' to include all 
legs = 'all' # [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11] 
voxel_sizes = 'all' # 'all' # [0.2, 0.5, 1.0, 2.0]

# Set the average leaf area
average_leaf_area = 0.0024149995507188344  # in m^2, adjust as needed

# Get the list of all voxel sizes
intersection_files = []
if legs == 'all' and voxel_sizes == 'all':
    intersection_files = glob.glob(os.path.join(valid_rays_dir, '*_intersections.parquet'))
elif legs == 'all' and isinstance(voxel_sizes, list):
    for voxel_size in voxel_sizes:
        intersection_files += glob.glob(os.path.join(valid_rays_dir, f'leg_*_voxel_{voxel_size}_intersections.parquet'))
elif isinstance(legs, list) and voxel_sizes == 'all':
    for leg in legs:
        intersection_files += glob.glob(os.path.join(valid_rays_dir, f'leg_{leg}_*_intersections.parquet'))
else:
    for leg in legs:
        for voxel_size in voxel_sizes:
            intersection_files += glob.glob(os.path.join(valid_rays_dir, f'leg_{leg}_voxel_{voxel_size}_intersections.parquet'))

# Check if any intersection files were found
if intersection_files == []:
    print("No intersection files found. Please check the input parameters.")

# Split intersection files into separate lists for each voxel_size
voxel_size_files = {}
for file in intersection_files:
    # Extract the voxel size from the filename
    parts = file.split('_')
    voxel_size = float(parts[parts.index('voxel') + 1])
    
    # Add the file to the corresponding voxel size list
    if voxel_size not in voxel_size_files:
        voxel_size_files[voxel_size] = []
    voxel_size_files[voxel_size].append(file)

# Extract voxel information for each voxel size
for voxel_size, files in voxel_size_files.items():
    # Create a list of all legs in files
    legs = []
    for file in files:
        leg = os.path.basename(file)
        parts = leg.split('_')
        leg = int(parts[parts.index('leg') + 1])
        legs.append(leg)

    # Calculate the lambda_1 for average leaf area
    lambda_1 = calculate_lambda_1(voxel_size=voxel_size, average_leaf_area=average_leaf_area)
    print(f"Voxel size: {voxel_size}, Lambda_1: {lambda_1}")

    # Calculate per voxel information from all files
    voxel_metrics_df = get_voxel_metrics(
        intersections_files=files, 
        lambda_1=lambda_1, 
        is_multireturn=True
    )

    # Retrieve the reference file
    reference_file = glob.glob(os.path.join(references_dir, f'*voxel_size_{voxel_size}*'))[0]
    df_ref = pd.read_csv(reference_file)

    # CI_leaf_Corr, CI_lw_Corr
    # Ensure only numeric columns are included in the mean operation
    df_ref = df_ref.groupby('voxel_id').mean(numeric_only=True).reset_index()
    df_ref = df_ref.add_suffix('_ref')


    df_ref.rename(columns={
        'voxel_id_ref': 'voxel_id',
        'LAD_ref_ref': 'LAD_ref', 
        'PAD_ref_ref': 'PAD_ref'
        }, inplace=True)

    # Merge to maintain voxel_id matching
    voxel_metrics_df = voxel_metrics_df.merge(df_ref, on='voxel_id', how='left')
    voxel_metrics_df.drop(columns=[
        "voxel_cx_ref",
        "voxel_cy_ref",
        "voxel_cz_ref"
    ])

    ### Add LAD calculations here if desired
    """Example, LAD_BL_TLS

    # Retrieve required variables
    I_leaf = voxel_metrics_df['I_leaf'].values
    mean_path_length = voxel_metrics_df['mean_path_length'].values  
    G_leaf = voxel_metrics_df['G_leaf'].values
    CI_leaf_ref = voxel_metrics_df['CI_leaf_corr_ref'].values

    LAD_BL_TLS = utils.BL_pimont_2018(I=I_leaf, mean_path_length=mean_path_length)
    LAD_BL_TLS_G = utils.BL_pimont_2018(I=I_leaf, mean_path_length=mean_path_length, G=G_leaf)
    LAD_BL_TLS_CI_ref = utils.BL_pimont_2018(I=I_leaf, mean_path_length=mean_path_length, G=G_leaf, CI=CI_leaf_ref)
    """

    # Save outputs to csv
    project_name = os.path.basename(os.path.normpath(project_dir))
    legs.sort()
    leg_string = "_".join(map(str, legs))
    output_file = os.path.join(results_dir, f"{project_name}_leg_{leg_string}_voxel_size_{voxel_size}.csv")
    if os.path.exists(output_file):
        os.remove(output_file)
    voxel_metrics_df.to_csv(output_file)

Voxel size: 2.0, Lambda_1: 0.0003018749438398543
[###########################             ] | 68% Completed | 334.32 s


UnboundLocalError: cannot access local variable 'echo_intensities' where it is not associated with a value